In [3]:
"""
VIX as Predictor of Next-Day SPY Realized Volatility
Layer 1: Statistical Association Test

Per the Evaluation Protocol (framework/evaluation_protocol.md):
- Target: SPY Garman-Klass annualized volatility
- Horizon: t+1 (next trading day)
- Threshold: p < 0.05 with sample size >= 250
"""

import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from scipy import stats
import matplotlib.pyplot as plt

# Repo root (hardcoded because VS Code notebooks launch from VS Code's
# install directory, not the workspace folder)
REPO_ROOT = Path(r"C:\Users\marcm\datascience\DS\Projects\market-research-lab")
load_dotenv(REPO_ROOT / ".env")

assert os.getenv("POSTGRES_DB"), "Could not locate .env — check REPO_ROOT path"
print(f"Repo root: {REPO_ROOT}")
print(f"Working with database: {os.getenv('POSTGRES_DB')}")

Repo root: C:\Users\marcm\datascience\DS\Projects\market-research-lab
Working with database: quant_research


In [4]:
def get_db_connection():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST"),
        port=os.getenv("POSTGRES_PORT"),
        dbname=os.getenv("POSTGRES_DB"),
        user=os.getenv("POSTGRES_USER"),
        password=os.getenv("POSTGRES_PASSWORD"),
    )


# Load VIX close prices and SPY GK volatility, joined on date
sql = """
    SELECT
        m.date,
        m.close AS vix_close,
        v.value AS spy_gk_vol
    FROM market_data m
    JOIN volatility_metrics v
      ON m.date = v.date
     AND v.ticker = 'SPY'
     AND v.metric_name = 'garman_klass_annualized_pct'
    WHERE m.ticker = '^VIX'
    ORDER BY m.date ASC
"""

with get_db_connection() as conn:
    df = pd.read_sql(sql, conn)

print(f"Loaded {len(df)} joined rows")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

Loaded 4112 joined rows
Date range: 2010-01-04 to 2026-05-08


C:\Users\marcm\AppData\Local\Temp\ipykernel_15092\106966008.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,date,vix_close,spy_gk_vol
0,2010-01-04,20.040001,16.785425
1,2010-01-05,19.350000,7.570115
2,2010-01-06,19.160000,5.276086
3,2010-01-07,19.059999,9.644474
4,2010-01-08,18.129999,7.391669


In [5]:
# Construct the predictor-target pair
# - Predictor: VIX close on day t
# - Target: SPY GK volatility on day t+1
#
# To align them, we shift the target column UP by 1 day, so each row
# now represents: "Today's VIX → tomorrow's volatility."
#
# This is the only correct way to construct a forecast pair without
# look-ahead bias.

df = df.sort_values("date").reset_index(drop=True)

df["spy_gk_vol_next"] = df["spy_gk_vol"].shift(-1)

# Drop the last row, which has no t+1 target available
df_pairs = df.dropna(subset=["spy_gk_vol_next"]).copy()

print(f"Original rows: {len(df)}")
print(f"After alignment (one row dropped, no t+1 available): {len(df_pairs)}")
print()
print("First 5 rows of the aligned dataset:")
print(df_pairs[["date", "vix_close", "spy_gk_vol", "spy_gk_vol_next"]].head())
print()
print("Last 5 rows:")
print(df_pairs[["date", "vix_close", "spy_gk_vol", "spy_gk_vol_next"]].tail())

Original rows: 4112
After alignment (one row dropped, no t+1 available): 4111

First 5 rows of the aligned dataset:
         date  vix_close  spy_gk_vol  spy_gk_vol_next
0  2010-01-04  20.040001   16.785425         7.570115
1  2010-01-05  19.350000    7.570115         5.276086
2  2010-01-06  19.160000    5.276086         9.644474
3  2010-01-07  19.059999    9.644474         7.391669
4  2010-01-08  18.129999    7.391669         8.176222

Last 5 rows:
            date  vix_close  spy_gk_vol  spy_gk_vol_next
4106  2026-05-01  16.990000    6.784919        10.773652
4107  2026-05-04  18.290001   10.773652         4.785524
4108  2026-05-05  17.379999    4.785524         7.031687
4109  2026-05-06  17.389999    7.031687         8.583497
4110  2026-05-07  17.080000    8.583497         3.954466
